# Imports

In [ ]:
!pip install torch_geometric

In [2]:
import os
import numpy as np
import pandas as pd
import copy
import pprint
import gc
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, LayerNorm as GraphLayerNorm
from torch_geometric.data import Dataset
from torch_geometric.loader import DataLoader

import joblib
from datetime import datetime, timezone, timedelta

from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score, average_precision_score, roc_auc_score, confusion_matrix, precision_recall_curve
from scipy.special import expit # It is the optimized sigmoid of Scipy

import random
import json
import matplotlib.pyplot as plt

In [1]:
# Change directory
os.chdir('/content/drive/MyDrive/nids-mitre/')

!pwd


/content/drive/MyDrive/nids-mitre


# Experiment Manager

In [4]:
class ExperimentManager:
    def __init__(self, log_file="./logs/experiments_log_gnn.csv", model_dir="./saved_models"):
        self.log_file = log_file
        self.model_dir = model_dir
        os.makedirs(os.path.dirname(log_file), exist_ok=True)
        os.makedirs(model_dir, exist_ok=True)

    def log_experiment(self, model_config=None, model_name=None, params=None, metrics=None, model_object=None):
        """
        Record an experiment in CSV format and optionally save the model.

        model_config: Recommended dict:
        - model_name (str)
        - type (str)
        - model_params (dict) # ONLY model hyperparameters
        - prob_threshold (float) # Decision threshold for probabilities (for metrics computation)
        - data_params (dict) # Optional
        - extra_params (dict) # Optional

        model_name: Name of the model (str)

        params: (legacy) Extra dict (for compatibility)
        metrics: Dict with results
        model_object: Model object (for saving)
        """

        if metrics is None:
            metrics = {}
        if params is None:
            params = {}

        # Timezone Argentina
        tz = timezone(timedelta(hours=-3))
        now = datetime.now(tz)

        # Input
        if model_config is not None:
            mname = model_config.get("model_name", model_name)
            mtype = model_config.get("type", None)
            model_params = model_config.get("model_params", {})
            threshold = model_config.get("prob_threshold", None)
            data_params = model_config.get("data_params", {})
            extra_params = model_config.get("extra_params", {})
        else:
            # legacy mode
            mname = model_name
            mtype = params.get("type", None)
            threshold = params.get("prob_threshold", None)
            model_params = params
            data_params = {}
            extra_params = {}

        # Create record
        entry = {
            "timestamp": now.strftime("%Y-%m-%d %H:%M:%S"),
            "model_name": mname,
        }
        if mtype is not None:
            entry["type"] = mtype
        if threshold is not None:
            entry["prob_threshold"] = threshold

        # Prefijos para evitar colisiones de nombres
        entry.update({f"hp_{k}": v for k, v in (model_params or {}).items()})
        entry.update({f"data_{k}": v for k, v in (data_params or {}).items()})
        entry.update({f"extra_{k}": v for k, v in {**extra_params, **params}.items()
                      if k not in ("type", "prob_threshold")})

        entry.update(metrics)

        # Save in csv
        df_new = pd.DataFrame([entry])
        if os.path.exists(self.log_file):
            df_new.to_csv(self.log_file, mode="a", header=False, index=False)
        else:
            df_new.to_csv(self.log_file, mode="w", header=True, index=False)

        print(f"Experiment recorded in {self.log_file}")

        # Save model
        if model_object is not None:
            metric_key = "AUC-PR" if "AUC-PR" in metrics else ("F1" if "F1" in metrics else None)
            metric_val = metrics.get(metric_key, 0) if metric_key else 0
            safe_key = metric_key or "metric"
            filename = f"{mname}_{now.strftime('%Y%m%d_%H%M')}_{safe_key}_{float(metric_val):.4f}"

            if "sklearn" in str(type(model_object)):
                import joblib
                joblib.dump(model_object, os.path.join(self.model_dir, f"{filename}.joblib"))
            else:
                import torch
                torch.save(model_object.state_dict(), os.path.join(self.model_dir, f"{filename}.pth"))

            print(f"Saved model: {filename}")


# Global instance
exp_manager = ExperimentManager()

# Load data

In [5]:
class NF_IDS_Dataset(Dataset):
    def __init__(self, root_dir, split='train'):
        # root_dir: (eg: "./dataset_processed")
        # split: 'train', 'val' or 'test'
        self.split_dir = os.path.join(root_dir, split)
        super().__init__(self.split_dir)

        # List files ordered numerically to respect the time
        self.files = sorted(
            [f for f in os.listdir(self.split_dir) if f.endswith('.pt')],
            key=lambda x: int(x.split('_')[1].split('.')[0])
        )

    def len(self):
        return len(self.files)

    def get(self, idx):
        data = torch.load(
            os.path.join(self.split_dir, self.files[idx]),
            weights_only=False # to allow loading complex graph objects
        )
        return data

# Models

## Static GNN Identity

In [ ]:
class StaticGNN_Identity(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim, dropout=0.2, output_bias_init=None):
        super(StaticGNN_Identity, self).__init__()

        self.hidden_dim = hidden_dim
        self.dropout_rate = dropout

        # IDENTITY PROJECTION
        # Layer to transform the raw edge attributes into a node vector
        self.edge_proj = nn.Linear(edge_dim, node_dim)

        # GNN INPUT DIMENSIONS
        # Since (What the IP sends + What the IP receives) will be concatenated, the input is double.
        gnn_input_dim = 2 * node_dim

        # GNN LAYER 1
        self.gnn1 = GATv2Conv(
            in_channels=gnn_input_dim,
            out_channels=hidden_dim,
            edge_dim=edge_dim,
            heads=2,
            concat=False,
            dropout=dropout
        )
        self.norm1 = GraphLayerNorm(hidden_dim) # to stabilize traffic features

        # GNN LAYER 2
        self.gnn2 = GATv2Conv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            edge_dim=edge_dim,
            heads=1,
            concat=False,
            dropout=dropout
        )
        self.norm2 = GraphLayerNorm(hidden_dim) # to stabilize traffic features

        # EDGE CLASSIFIER
        classifier_input_dim = (2 * hidden_dim) + edge_dim

        self.classifier = nn.Sequential(
            nn.Linear(classifier_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )

        # Bias Init
        if output_bias_init is not None:
            self.classifier[-1].bias.data.fill_(output_bias_init)

    # --- Utils for building "identity" (node features) ---
    def manual_scatter_mean(self, src, index, dim_size):
        out = torch.zeros((dim_size, src.size(1)), device=src.device)
        out.index_add_(0, index, src)
        ones = torch.ones(src.size(0), 1, device=src.device)
        count = torch.zeros(dim_size, 1, device=src.device)
        count.index_add_(0, index, ones)
        count[count < 1] = 1
        return out / count

    def forward(self, x, edge_index, edge_attr):
        # Check empty graph
        if x.size(0) == 0: return torch.empty((0, 1), device=x.device)

        # ---------------------------------------------------------
        # STEP 1: BUILDING "IDENTITY" (node features)
        # ---------------------------------------------------------
        edge_embeddings = self.edge_proj(edge_attr)
        edge_embeddings = F.relu(edge_embeddings)

        src, dst = edge_index
        num_nodes = x.size(0)

        # Dual Identity
        x_out = self.manual_scatter_mean(edge_embeddings, src, dim_size=num_nodes)
        x_in  = self.manual_scatter_mean(edge_embeddings, dst, dim_size=num_nodes)

        # Concatenate [What IP sends | What IP recibes]
        x_input = torch.cat([x_out, x_in], dim=1)

        # ---------------------------------------------------------
        # STEP 2: GNN LAYERS
        # ---------------------------------------------------------

        # Layer 1
        h1 = self.gnn1(x_input, edge_index, edge_attr=edge_attr)
        h1 = self.norm1(h1) # LayerNorm
        h1 = F.elu(h1)
        h1 = F.dropout(h1, p=self.dropout_rate, training=self.training)

        # Layer 2
        h2 = self.gnn2(h1, edge_index, edge_attr=edge_attr)
        h2 = self.norm2(h2) # LayerNorm
        h2 = F.elu(h2)

        # ---------------------------------------------------------
        # STEP 3: CLASSIFICATION
        # ---------------------------------------------------------
        src, dst = edge_index
        edge_rep = torch.cat([h2[src], h2[dst], edge_attr], dim=1)

        return self.classifier(edge_rep)

## ST_GNN_Identity

In [6]:
class ST_GNN_Identity(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim, dropout=0.2, output_bias_init=None):
        super(ST_GNN_Identity, self).__init__()

        self.hidden_dim = hidden_dim
        self.dropout_rate = dropout

        # IDENTITY PROJECTION
        # Layer to transform the raw edge attributes into a node vector
        self.edge_proj = nn.Linear(edge_dim, node_dim)

        # GNN INPUT DIMENSIONS
        # Since (What the IP sends + What the IP receives) will be concatenated, the input is double.
        gnn_input_dim = 2 * node_dim

        # GNN LAYER 1
        self.gnn1 = GATv2Conv(
            in_channels=gnn_input_dim,
            out_channels=hidden_dim,
            edge_dim=edge_dim,
            heads=2,
            concat=False,
            dropout=dropout
        )
        self.norm1 = GraphLayerNorm(hidden_dim) # to stabilize traffic features

        # GNN LAYER 2
        self.gnn2 = GATv2Conv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            edge_dim=edge_dim,
            heads=1,
            concat=False,
            dropout=dropout
        )
        self.norm2 = GraphLayerNorm(hidden_dim) # to stabilize traffic features

        # TEMPORAL (GRU)
        self.gru = nn.GRUCell(input_size=hidden_dim, hidden_size=hidden_dim)

        # EDGE CLASSIFIER
        classifier_input_dim = (2 * hidden_dim) + edge_dim

        self.classifier = nn.Sequential(
            nn.Linear(classifier_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )

        # Bias Init
        if output_bias_init is not None:
            self.classifier[-1].bias.data.fill_(output_bias_init)

        self.node_memory = {}

    # --- UTILS ---
    def manual_scatter_mean(self, src, index, dim_size):
        out = torch.zeros((dim_size, src.size(1)), device=src.device)
        out.index_add_(0, index, src)

        ones = torch.ones(src.size(0), 1, device=src.device)
        count = torch.zeros(dim_size, 1, device=src.device)
        count.index_add_(0, index, ones)

        count[count < 1] = 1
        return out / count

    def get_memory(self, ids, device):
        mem_list = []
        for i in ids:
            i = i.item()
            if i in self.node_memory:
                mem_list.append(self.node_memory[i])
            else:
                mem_list.append(torch.zeros(self.hidden_dim, device=device))
        return torch.stack(mem_list)

    def update_memory(self, ids, h_new):
        h_detached = h_new.detach()
        for idx, global_id in enumerate(ids):
            self.node_memory[global_id.item()] = h_detached[idx]

    def detach_all_memory(self):
        for k in self.node_memory:
            self.node_memory[k] = self.node_memory[k].detach()

    def reset_memory(self):
        self.node_memory = {}

    def forward(self, x, edge_index, edge_attr, global_ids):
        if x.size(0) == 0: return torch.empty((0, 1), device=x.device)

        # ---------------------------------------------------------
        # STEP 1: BUILDING "IDENTITY" (node features)
        # ---------------------------------------------------------

        edge_embeddings = self.edge_proj(edge_attr)
        edge_embeddings = F.relu(edge_embeddings) # Activación inicial

        src, dst = edge_index
        num_nodes = x.size(0)

        # Dual Identity
        x_out = self.manual_scatter_mean(edge_embeddings, src, dim_size=num_nodes)
        x_in  = self.manual_scatter_mean(edge_embeddings, dst, dim_size=num_nodes)

        # Concatenate [What IP sends | What IP recibes]
        # Size: [Num_Nodes, 2 * Node_Dim]
        x_input = torch.cat([x_out, x_in], dim=1)

        # ---------------------------------------------------------
        # STEP 2: GNN PROCESSING + MEMORY
        # ---------------------------------------------------------
        h_prev = self.get_memory(global_ids, x.device)

        # GNN LAYER 1
        z = self.gnn1(x_input, edge_index, edge_attr=edge_attr)
        z = self.norm1(z) # LayerNorm
        z = F.elu(z)
        z = F.dropout(z, p=self.dropout_rate, training=self.training)

        # GNN LAYER 2
        z = self.gnn2(z, edge_index, edge_attr=edge_attr)
        z = self.norm2(z)
        z = F.elu(z)

        # GRU LAYER
        h_current = self.gru(z, h_prev)
        self.update_memory(global_ids, h_current)

        # ---------------------------------------------------------
        # STEP 3: CLASSIFICATION
        # ---------------------------------------------------------
        edge_rep = torch.cat([h_current[src], h_current[dst], edge_attr], dim=1)

        return self.classifier(edge_rep)

# Functions

This handles the complex logic: Truncated Backpropagation for ST-GNN and detailed metric calculations (including False Positives).

## Metrics

In [7]:
def calculate_metrics_gnn(y_true, y_pred_logits, prob_threshold=0.5):
    """
    y_true: List or array of actual labels (0 or 1).
    y_pred_logits: List or array of raw model outputs (before sigmoid).
    """
    y_true = np.array(y_true)
    logits = np.array(y_pred_logits)

    # Convert logits to probabilities, and to classes (0 or 1) depend on prob_threshold
    probs = expit(logits)
    preds = (probs > prob_threshold).astype(int)

    # Threshold-dependent metrics
    prec = precision_score(y_true, preds, zero_division=0)
    rec = recall_score(y_true, preds, zero_division=0)
    f1 = f1_score(y_true, preds, zero_division=0)
    f2 = fbeta_score(y_true, preds, beta=2, zero_division=0)

    # Threshold-independent metrics
    try:
        ap = average_precision_score(y_true, probs)
        roc = roc_auc_score(y_true, probs)
    except ValueError:
        # This occurs if y_true has only one class (e.g., only benign in the batch)
        ap = 0.0
        roc = 0.5

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    return {
        "Precision": prec, "Recall": rec, "F1": f1, "F2": f2, "AUC-PR": ap, "AUC-ROC": roc,
        "FPR": fpr, "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn), "Total_Flows": len(y_true)
    }



## Train and eval

In [8]:
def train_epoch(model, loader, optimizer, criterion, device, is_temporal=False, batch_steps=10):
    model.train()
    total_loss = 0
    steps = 0  # Count of valid graphs processed (not empty)

    # Loss accumulator for Truncated Backprop
    batch_loss = 0

    # Iterate over the Loader
    # batch_idx helps to know if we are on the last element
    for batch_idx, data in enumerate(loader):
        data = data.to(device)

        # If it is an empty graph, skip
        if data.x.shape[0] == 0:
            continue

        # Forward
        if is_temporal:
            out = model(data.x, data.edge_index, data.edge_attr, data.global_node_ids)
        else:
            out = model(data.x, data.edge_index, data.edge_attr)

        # Loss calculation
        loss = criterion(out.view(-1), data.y)
        batch_loss += loss
        steps += 1

        # Backpropagation:
        # 1. Each 'batch_steps' valid steps (e.g., every 10 graphs)
        # 2. Or if it is in the LAST batch of the loader (so as not to lose the remainder)
        is_last_batch = (batch_idx == len(loader) - 1)

        if (steps > 0 and steps % batch_steps == 0) or is_last_batch:
            # Only update if there's something accumulated
            if batch_loss > 0:
                optimizer.zero_grad()
                batch_loss.backward()
                optimizer.step()

                # Reset
                total_loss += batch_loss.item()
                batch_loss = 0

                # IMPORTANT for ST-GNN: Detach memory
                if is_temporal:
                    model.detach_all_memory()

    # Average loss per valid step
    return total_loss / steps if steps > 0 else 0

@torch.no_grad()
def evaluate(model, loader, criterion, device, prob_threshold, is_temporal=False):
    model.eval()
    all_logits = []
    all_trues = []
    total_loss = 0
    steps = 0

    # Clear the memory before starting the evaluation (only if it's temporal)
    if is_temporal:
        model.node_memory = {}

    # Don't need enumerate here because we're not doing backprop
    for data in loader:
        data = data.to(device)

        if data.x.shape[0] == 0: continue

        if is_temporal:
            out = model(data.x, data.edge_index, data.edge_attr, data.global_node_ids)
        else:
            out = model(data.x, data.edge_index, data.edge_attr)

        loss = criterion(out.view(-1), data.y)
        total_loss += loss.item()

        all_logits.extend(out.view(-1).cpu().numpy())
        all_trues.extend(data.y.cpu().numpy())
        steps += 1

    metrics = calculate_metrics_gnn(all_trues, all_logits, prob_threshold)
    metrics["Loss"] = total_loss / steps if steps > 0 else 0
    return metrics

## Optimal threshold

In [9]:
def find_optimal_threshold(model, loader, device, is_temporal=False):
    model.eval()

    if is_temporal:
        # Clear the memory before starting the evaluation (only if it's temporal)
        model.node_memory = {}

    all_probs = []
    all_trues = []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)

            if is_temporal:
                out = model(data.x, data.edge_index, data.edge_attr, data.global_node_ids)
            else:
                out = model(data.x, data.edge_index, data.edge_attr)

            # Convert Logits to Probabilities (Sigmoid)
            probs = torch.sigmoid(out.view(-1))

            all_probs.extend(probs.cpu().numpy())
            all_trues.extend(data.y.cpu().numpy())

    all_trues = np.array(all_trues)
    all_probs = np.array(all_probs)

    # 1. Precision-Recall Curve
    precisions, recalls, thresholds = precision_recall_curve(all_trues, all_probs)

    # 2. Calculate F1 for each possible threshold
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
    f1_scores = f1_scores[:-1]

    # 3. Find max
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]
    best_rec = recalls[best_idx]
    best_prec = precisions[best_idx]

    print(f"\nBEST THRESHOLD FOUND: {best_threshold:.4f}")
    print(f"   F1 Score : {best_f1:.4f}")
    print(f"   Recall   : {best_rec:.4f}")
    print(f"   Precision: {best_prec:.4f}")

    # Probability Diagnosis
    avg_benign = np.mean(all_probs[all_trues == 0])
    avg_attack = np.mean(all_probs[all_trues == 1])
    print(f"\nProbability Diagnosis:")
    print(f"   Average assigned to Benignos: {avg_benign:.4f}")
    print(f"   Average assigned to Attacks : {avg_attack:.4f}")

    return best_threshold

## Run multiple seeds

In [10]:
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NumpyEncoder, self).default(obj)

In [11]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)

In [12]:
def run_multiple_seeds(model_class, model_config, train_loader, val_loader,
                       manager,
                       seeds=[42, 100, 2024, 777, 99],
                       epochs=60,
                       device='cpu',
                       experiment_name="Experiment"):

    results_summary = {
        'seeds': [],
        'best_aucpr': [],
        'best_f1': [],
        'best_threshold': []
    }

    all_losses = {}

    print(f" Starting Multi-Seed Run: {experiment_name}")
    print(f"   Seeds: {seeds}")
    print("-" * 60)

    for seed in seeds:
        # Name
        exp_id = experiment_name + f"_seed{seed}"
        print(f"\n{exp_id}")

        # Preventive Memory Cleaning (Before loading anything)
        gc.collect()
        torch.cuda.empty_cache()

        # Update model_config
        model_config['model_name'] = exp_id

        print(f"\n Seed: {seed} ...")
        set_seed(seed)

        # 1. Instantiate Model
        model = model_class(**model_config['model_params']).to(device)

        # 2. Training Setup
        optimizer = torch.optim.Adam(model.parameters(), lr=model_config['extra_params']['learning_rate'])
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([model_config['extra_params']['pos_weight']]).to(device))

        # Variables for tracking within this seed
        train_loss_history = []
        val_loss_history = []

        best_auc_seed = 0
        best_metrics_seed = {}
        best_model_wts = copy.deepcopy(model.state_dict())

        # Epochs
        for epoch in range(1, epochs + 1):
            # Detect whether it is a temporal model (ST or GRU) to adjust inputs
            is_temporal = 'GRU' in experiment_name or 'ST' in experiment_name

            # --- TRAIN ---
            loss_train = train_epoch(model, train_loader, optimizer, criterion,
                                     device=device, is_temporal=is_temporal,
                                     batch_steps=model_config['extra_params']["batch_steps"])

            # --- EVAL ---
            metrics = evaluate(model, val_loader, criterion, device,
                               prob_threshold=model_config['prob_threshold'], is_temporal=is_temporal)

            # Save history
            train_loss_history.append(float(loss_train))
            val_loss_history.append(float(metrics['Loss']))

            if (epoch+1) % 10 == 0: # Print every 10 epochs
                print(f"   Ep {epoch+1} | Loss: {loss_train:.4f} | Val AUC-PR: {metrics['AUC-PR']:.4f} | Val Rec: {metrics['Recall']:.4f}")

            # 4. Checkpoint of the best model of THIS seed
            if metrics['AUC-PR'] > best_auc_seed:
                best_auc_seed = metrics['AUC-PR']
                best_metrics_seed = metrics
                best_model_wts = copy.deepcopy(model.state_dict())

        # 5. LOGGING WITH EXPERIMENT MANAGER
        model.load_state_dict(best_model_wts)
        manager.log_experiment(
            model_config=model_config,
            metrics=best_metrics_seed,
            model_object=model
        )

        print(f"\nBEST AUC-PR: {best_auc_seed:.4f} (FPR: {metrics['FPR']:.4f})")
        best_threshold_seed = find_optimal_threshold(model, val_loader, device, is_temporal)

        # 6. Accumulate for JSON report
        results_summary['seeds'].append(seed)
        results_summary['best_aucpr'].append(float(best_auc_seed))
        results_summary['best_f1'].append(float(best_metrics_seed.get('F1', 0)))
        results_summary['best_threshold'].append(float(best_threshold_seed))

        all_losses[f"seed_{seed}"] = {
            'train': train_loss_history,
            'val': val_loss_history
        }

        # 7. Save JSON
        filename_json = f"./logs/curves_multiseed_{experiment_name}.json"
        try:
            with open(filename_json, 'w') as f:
                json.dump({'summary': results_summary, 'losses': all_losses}, f, cls=NumpyEncoder)
            print(f"\nProgressive backup of losses saved in: {filename_json}")
        except Exception as e:
            print(f"\nWarning: JSON backup could not be saved: {e}")

        print(f"\n End seed {seed}")

        print("-" * 60)

    # 8. Print final statistics
    avg_auc = np.mean(results_summary['best_aucpr'])
    std_auc = np.std(results_summary['best_aucpr'])

    print("\n" + "="*50)
    print(f" AVERAGE RESULT: {experiment_name}")
    print(f"   AUC-PR: {avg_auc:.4f} ± {std_auc:.4f}")
    print("="*50)


# Main

In [13]:
ROOT_PATH = "./dataset_processed"

In [14]:
# Instantiate Dataset (Only reads file names)
train_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='train')
val_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='val')

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

# Instantiate DataLoader (Load manager)
# batch_size=1 : Important for ST-GNN to handle memory step by step
# num_workers=2 : Use 2 CPU cores to load files while training
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=False) # pin_memory=False for CPU
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=False)

Train size: 1998 | Val size: 428


## First run

### STATIC: Bias init - LayerNorm - Identity (STATIC)

In [ ]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

EPOCHS = 60
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005
POS_WEIGHT = 2.0

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
#HIDDEN_DIM = 32
DROPOUT = 0.2
BIAS_VALUE = -2.9968

PROB_THRESHOLD = 0.5

search_space = {
    'hidden_dim': [32]#, 64]
}


Using device: cpu


In [ ]:
model_config = {
    "model_name": "Static_GNN_biasinit",
    "type": "GNN_static_BiasOn",
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": None,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE
    },
    "prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": POS_WEIGHT,
        "learning_rate": LR
    }
}

In [ ]:
exp_manager = ExperimentManager(log_file="./logs/experiments_log_gnn_biasinit_robust_identity.csv")

In [ ]:
# Load existing history to avoid repeating
existing_experiments = []
if os.path.exists(exp_manager.log_file):
    try:
        df_log = pd.read_csv(exp_manager.log_file)
        if 'model_name' in df_log.columns:
            existing_experiments = df_log['model_name'].values.tolist()
        print(f"History loaded. {len(existing_experiments)} experiments already performed")
    except:
        print("The current log could not be read, we will start from scratch")

count = 0

for h_dim in search_space['hidden_dim']:
    count += 1

    # Name
    exp_id = f"Static_GNN_H{h_dim}_BiasOn_robust_identity"
    print(f"\n{exp_id}")

    if exp_id in existing_experiments:
        print(f" Skipping {exp_id} (Already registered in CSV)")
        continue

    print(f"\n Starting: {exp_id}")

    # Preventive Memory Cleaning (Before loading anything)
    gc.collect()
    torch.cuda.empty_cache()

    # Update model_config
    model_config['model_name'] = exp_id
    model_config["model_params"]["hidden_dim"] = h_dim

    try:
        # Instantiate
        model = StaticGNN_Identity(**model_config['model_params']).to(DEVICE)

        # Configure training
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT]).to(DEVICE))

        # Deepcopy vars
        best_aucpr = 0.0
        best_wts = copy.deepcopy(model.state_dict())
        best_metrics = {}

        for epoch in range(EPOCHS):
            loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE, is_temporal=False, batch_steps=BATCH_STEPS)
            metrics = evaluate(model, val_loader, criterion, DEVICE, prob_threshold=PROB_THRESHOLD, is_temporal=False)

            if (epoch+1) % 2 == 0: # Print every 2 epochs
                print(f"   Ep {epoch+1} | Loss: {loss:.4f} | Val AUC-PR: {metrics['AUC-PR']:.4f} | Val Rec: {metrics['Recall']:.4f}")

            if metrics['AUC-PR'] > best_aucpr:
                best_aucpr = metrics['AUC-PR']
                best_metrics = metrics
                best_wts = copy.deepcopy(model.state_dict())

        # Restore and save
        model.load_state_dict(best_wts)
        exp_manager.log_experiment(model_config=model_config, metrics=best_metrics, model_object=model)
        print(f"   Best AUC-PR: {best_aucpr:.4f} (FPR: {metrics['FPR']:.4f})")
        _=find_optimal_threshold(model, val_loader, DEVICE, False)
        print("=============\n")

    except Exception as e:
        print(f"   FATAL ERROR in {exp_id}: {str(e)}")
        continue

    # Clean memory
    del model
    del optimizer
    del criterion
    del best_wts
    gc.collect()
    torch.cuda.empty_cache()




### ST: Bias init - LayerNorm - Identity

In [ ]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

EPOCHS = 60
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005
POS_WEIGHT = 2.0

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
#HIDDEN_DIM = 32
DROPOUT = 0.2
BIAS_VALUE = -2.9968

PROB_THRESHOLD = 0.5

search_space = {
    'hidden_dim': [32]#, 64]
}


Using device: cpu


In [ ]:
model_config = {
    "model_name": "ST_GNN_biasinit",
    "type": "GNN_and_GRU_BiasOn",
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": None,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE
    },
    "prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": POS_WEIGHT,
        "learning_rate": LR
    }
}

In [ ]:
exp_manager = ExperimentManager(log_file="./logs/experiments_log_gnn_biasinit_robust_identity.csv")

In [ ]:
# Load existing history to avoid repeating
existing_experiments = []
if os.path.exists(exp_manager.log_file):
    try:
        df_log = pd.read_csv(exp_manager.log_file)
        if 'model_name' in df_log.columns:
            existing_experiments = df_log['model_name'].values.tolist()
        print(f"History loaded. {len(existing_experiments)} experiments already performed")
    except:
        print("The current log could not be read, we will start from scratch")

count = 0

for h_dim in search_space['hidden_dim']:
    count += 1

    # Name
    exp_id = f"ST_GNN_H{h_dim}_BiasOn_robust_identity"
    print(f"\n{exp_id}")

    if exp_id in existing_experiments:
        print(f" Skipping {exp_id} (Already registered in CSV)")
        continue

    print(f"\n Starting: {exp_id}")

    # Preventive Memory Cleaning (Before loading anything)
    gc.collect()
    torch.cuda.empty_cache()

    # Update model_config
    model_config['model_name'] = exp_id
    model_config["model_params"]["hidden_dim"] = h_dim

    try:
        # Instantiate
        model = ST_GNN_Identity(**model_config['model_params']).to(DEVICE)

        # Configure training
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT]).to(DEVICE))

        # Deepcopy vars
        best_aucpr = 0.0
        best_wts = copy.deepcopy(model.state_dict())
        best_metrics = {}

        for epoch in range(EPOCHS):
            loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE, is_temporal=True, batch_steps=BATCH_STEPS)
            metrics = evaluate(model, val_loader, criterion, DEVICE, prob_threshold=PROB_THRESHOLD, is_temporal=True)

            if (epoch+1) % 2 == 0: # Print every 2 epochs
                print(f"   Ep {epoch+1} | Loss: {loss:.4f} | Val AUC-PR: {metrics['AUC-PR']:.4f} | Val Rec: {metrics['Recall']:.4f}")

            if metrics['AUC-PR'] > best_aucpr:
                best_aucpr = metrics['AUC-PR']
                best_metrics = metrics
                best_wts = copy.deepcopy(model.state_dict())

        # Restore and save
        model.load_state_dict(best_wts)
        exp_manager.log_experiment(model_config=model_config, metrics=best_metrics, model_object=model)
        print(f"   Best AUC-PR: {best_aucpr:.4f} (FPR: {metrics['FPR']:.4f})")
        _=find_optimal_threshold(model, val_loader, DEVICE, True)
        print("=============\n")

    except Exception as e:
        print(f"   FATAL ERROR in {exp_id}: {str(e)}")
        continue

    # Clean memory
    del model
    del optimizer
    del criterion
    del best_wts
    gc.collect()
    torch.cuda.empty_cache()




## Seeds

### StaticGNN_Identity

In [ ]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

EPOCHS = 60
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005
POS_WEIGHT = 2.0

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
HIDDEN_DIM = 32
DROPOUT = 0.2
BIAS_VALUE = -2.9968

PROB_THRESHOLD = 0.5



Using device: cpu


In [ ]:
model_config = {
    "model_name": None,
    "type": "SaticGNN_Identity",
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE
    },
    "prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": POS_WEIGHT,
        "learning_rate": LR
    }
}

In [ ]:
exp_manager = ExperimentManager(log_file="./logs/static_gnn_biasinit_robust_identity.csv", model_dir="./saved_models_static_gnn")

In [ ]:
run_multiple_seeds(
    model_class=StaticGNN_Identity,
    model_config=model_config,
    train_loader=train_loader,
    val_loader=val_loader,
    manager=exp_manager,
    seeds=[42, 123, 777, 2024, 99],
    epochs=EPOCHS,
    device=DEVICE,
    experiment_name="StaticGNN_BiasOn_robust_Identity"
)

 Starting Multi-Seed Run: StaticGNN_BiasOn_robust_Identity
   Seeds: [42, 123, 777, 2024, 99]
------------------------------------------------------------

StaticGNN_BiasOn_robust_Identity_seed42

 Seed: 42 ...
   Ep 10 | Loss: 0.1839 | Val AUC-PR: 0.1221 | Val Rec: 0.0000
   Ep 20 | Loss: 0.1657 | Val AUC-PR: 0.3323 | Val Rec: 0.1487
   Ep 30 | Loss: 0.1557 | Val AUC-PR: 0.3515 | Val Rec: 0.1529
   Ep 40 | Loss: 0.1523 | Val AUC-PR: 0.3857 | Val Rec: 0.1652
   Ep 50 | Loss: 0.1469 | Val AUC-PR: 0.3930 | Val Rec: 0.1691
   Ep 60 | Loss: 0.1454 | Val AUC-PR: 0.4568 | Val Rec: 0.2153
Experiment recorded in ./logs/static_gnn_biasinit_robust_identity.csv
Saved model: StaticGNN_BiasOn_robust_Identity_seed42_20260206_1143_AUC-PR_0.4636

BEST AUC-PR: 0.4636 (FPR: 0.0083)

BEST THRESHOLD FOUND: 0.3548
   F1 Score : 0.4657
   Recall   : 0.4228
   Precision: 0.5183

Probability Diagnosis:
   Average assigned to Benignos: 0.0626
   Average assigned to Attacks : 0.3786

Progressive backup oflosses

### ST_GNN_Identity

In [15]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

EPOCHS = 60
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005
POS_WEIGHT = 2.0

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
HIDDEN_DIM = 32
DROPOUT = 0.2
BIAS_VALUE = -2.9968

PROB_THRESHOLD = 0.5



Using device: cpu


In [16]:
model_config = {
    "model_name": None,
    "type": "ST_GNN_Identity",
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE
    },
    "prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": POS_WEIGHT,
        "learning_rate": LR
    }
}

In [17]:
exp_manager = ExperimentManager(log_file="./logs/st_gnn_biasinit_robust_identity.csv", model_dir="./saved_models_st_gnn")

In [ ]:
run_multiple_seeds(
    model_class=ST_GNN_Identity,
    model_config=model_config,
    train_loader=train_loader,
    val_loader=val_loader,
    manager=exp_manager,
    seeds=[42, 123, 777, 2024, 99],
    epochs=EPOCHS,
    device=DEVICE,
    experiment_name="ST_GNN_BiasOn_robust_Identity"
)

 Starting Multi-Seed Run: ST_GNN_BiasOn_robust_Identity
   Seeds: [42, 123, 777, 2024, 99]
------------------------------------------------------------

ST_GNN_BiasOn_robust_Identity_seed42

 Seed: 42 ...
   Ep 10 | Loss: 0.1453 | Val AUC-PR: 0.2370 | Val Rec: 0.0000
   Ep 20 | Loss: 0.1306 | Val AUC-PR: 0.4948 | Val Rec: 0.4336
   Ep 30 | Loss: 0.0725 | Val AUC-PR: 0.6473 | Val Rec: 0.6588
   Ep 40 | Loss: 0.0629 | Val AUC-PR: 0.6605 | Val Rec: 0.5181
   Ep 50 | Loss: 0.0812 | Val AUC-PR: 0.4522 | Val Rec: 0.3822
   Ep 60 | Loss: 0.0606 | Val AUC-PR: 0.4727 | Val Rec: 0.4075
Experiment recorded in ./logs/st_gnn_biasinit_robust_identity.csv
Saved model: ST_GNN_BiasOn_robust_Identity_seed42_20260206_1731_AUC-PR_0.6713

BEST AUC-PR: 0.6713 (FPR: 0.0138)

BEST THRESHOLD FOUND: 0.7112
   F1 Score : 0.6721
   Recall   : 0.6694
   Precision: 0.6747

Probability Diagnosis:
   Average assigned to Benignos: 0.0772
   Average assigned to Attacks : 0.7110

Progressive backup of losses saved in: .

In [19]:
# It stopped, so I ran it again for the last two seeds
run_multiple_seeds(
    model_class=ST_GNN_Identity,
    model_config=model_config,
    train_loader=train_loader,
    val_loader=val_loader,
    manager=exp_manager,
    seeds=[2024, 99],
    epochs=EPOCHS,
    device=DEVICE,
    experiment_name="ST_GNN_BiasOn_robust_Identity"
)

 Starting Multi-Seed Run: ST_GNN_BiasOn_robust_Identity
   Seeds: [2024, 99]
------------------------------------------------------------

ST_GNN_BiasOn_robust_Identity_seed2024

 Seed: 2024 ...
   Ep 10 | Loss: 0.1997 | Val AUC-PR: 0.1747 | Val Rec: 0.0000
   Ep 20 | Loss: 0.1456 | Val AUC-PR: 0.3306 | Val Rec: 0.0000
   Ep 30 | Loss: 0.1077 | Val AUC-PR: 0.6465 | Val Rec: 0.3569
   Ep 40 | Loss: 0.0971 | Val AUC-PR: 0.6155 | Val Rec: 0.4744
   Ep 50 | Loss: 0.0714 | Val AUC-PR: 0.6204 | Val Rec: 0.3591
   Ep 60 | Loss: 0.0725 | Val AUC-PR: 0.6509 | Val Rec: 0.4644
Experiment recorded in ./logs/st_gnn_biasinit_robust_identity.csv
Saved model: ST_GNN_BiasOn_robust_Identity_seed2024_20260207_0307_AUC-PR_0.7361

BEST AUC-PR: 0.7361 (FPR: 0.0053)

BEST THRESHOLD FOUND: 0.3091
   F1 Score : 0.7616
   Recall   : 0.8105
   Precision: 0.7182

Probability Diagnosis:
   Average assigned to Benignos: 0.0289
   Average assigned to Attacks : 0.5356

Progressive backup of losses saved in: ./logs/cu

Note: The above average (0.7847) was calculated using only the results from the last two seeds (part2).

In [8]:
!cat ./logs/curves_multiseed_ST_GNN_BiasOn_robust_Identity_PART1.json


{"summary": {"seeds": [42, 123, 777], "best_aucpr": [0.6712590354446151, 0.6582992344601789, 0.7587459071914067], "best_f1": [0.654213603308142, 0.610246249646193, 0.6830161607008005], "best_threshold": [0.7111920118331909, 0.4402283728122711, 0.36351361870765686]}, "losses": {"seed_42": {"train": [0.2266229599456793, 0.20072368976908508, 0.19108034355205492, 0.18374404945605374, 0.1550557547874269, 0.1508937226473776, 0.20007850606674904, 0.1606020947891372, 0.14533222523981731, 0.18333116884045939, 0.13611256704223923, 0.13828005891112163, 0.12597749751849338, 0.13218440931980618, 0.18244741384659585, 0.17886437732192753, 0.1439221679341934, 0.13295188221135684, 0.13062215539652478, 0.09745557508804638, 0.09467184370562297, 0.12885197613672225, 0.10343617319194269, 0.07957146036758059, 0.07650673939906423, 0.07555211506112089, 0.07025492300295348, 0.06901751718965436, 0.0725338606206081, 0.08435758419546609, 0.07082535714965373, 0.08030083789532996, 0.07351712937743382, 0.06918089766

In [9]:
!cat ./logs/curves_multiseed_ST_GNN_BiasOn_robust_Identity_PART2.json

{"summary": {"seeds": [2024, 99], "best_aucpr": [0.7360589970003276, 0.8332598840284609], "best_f1": [0.6529076439711775, 0.7914316476240981], "best_threshold": [0.3090564012527466, 0.4937387704849243]}, "losses": {"seed_2024": {"train": [0.20451679463786915, 0.1924876338274814, 0.20094617025415154, 0.20548284704871622, 0.20354573663603148, 0.19091088358072977, 0.21090870826418234, 0.1822967778073817, 0.19965192977138324, 0.188113945532644, 0.17857429334396954, 0.1735348081052225, 0.15370442962540748, 0.15511921884274202, 0.13348655804071413, 0.13248950658113978, 0.11191472456192061, 0.11816168368430552, 0.14556653788294913, 0.16753124812331527, 0.1217384165919015, 0.12141581936941315, 0.13697696304127438, 0.11160086717612004, 0.11260683529149941, 0.10619113400625003, 0.10623808054720955, 0.09954318176323497, 0.10765818081448605, 0.09278743035611592, 0.09477431064076311, 0.08950430010910465, 0.0837696186987666, 0.07473039781531059, 0.07850728522003422, 0.08524514657824481, 0.1135509872

In [18]:
import json
import numpy as np

file_part1 = "./logs/curves_multiseed_ST_GNN_BiasOn_robust_Identity_PART1.json"
file_part2 = "./logs/curves_multiseed_ST_GNN_BiasOn_robust_Identity_PART2.json"
output_file = "./logs/curves_multiseed_ST_GNN_BiasOn_robust_Identity_MERGED.json"

def merge_json_results(f1_path, f2_path, out_path, experiment_name):
    with open(f1_path, 'r') as f: data1 = json.load(f)
    with open(f2_path, 'r') as f: data2 = json.load(f)

    merged_data = {
        'summary': {
            'seeds': [],
            'best_aucpr': [],
            'best_f1': [],
            'best_threshold': []
        },
        'losses': {}
    }

    # 1. Merge Summary Lists
    keys = ['seeds', 'best_aucpr', 'best_f1', 'best_threshold']
    for k in keys:
        merged_data['summary'][k] = data1['summary'][k] + data2['summary'][k]

    # 2. Merge Loss Dictionaries
    merged_data['losses'].update(data1['losses'])
    merged_data['losses'].update(data2['losses'])

    # 3. Save final file
    with open(out_path, 'w') as f:
        json.dump(merged_data, f)

    print(f"File saved in: {out_path}")
    print(f"Total seeds: {merged_data['summary']['seeds']}")

    # 4. Print final statistics
    avg_auc = np.mean(merged_data['summary']['best_aucpr'])
    std_auc = np.std(merged_data['summary']['best_aucpr'])

    print("\n" + "="*50)
    print(f" AVERAGE RESULT: {experiment_name}")
    print(f"   AUC-PR: {avg_auc:.4f} ± {std_auc:.4f}")
    print("="*50)


merge_json_results(file_part1, file_part2, output_file, "ST_GNN_BiasOn_robust_Identity")

File saved in: ./logs/curves_multiseed_ST_GNN_BiasOn_robust_Identity_MERGED.json
Total seeds: [42, 123, 777, 2024, 99]

 AVERAGE RESULT: ST_GNN_BiasOn_robust_Identity
   AUC-PR: 0.7315 ± 0.0634
